### Read xlsx, create pandas dataframe, select definition of each tactics, embed by LLM, and save to csv

In [2]:
tactics_file = "enterprise-attack-v19.1-tactics.xlsx"
techniques_file = "enterprise-attack-v19.1-techniques.xlsx"

tactics_embeddings_file = "tactics_embeddings.csv"
techniques_embeddings_file = "techniques_embeddings.csv"

llm_model = "nomic-ai/nomic-embed-text-v2-moe"

In [6]:
# test gpu, tpu
import torch
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU is available
Using device: cuda


In [27]:
import pandas as pd
import numpy as np
tactics_df = pd.read_excel(tactics_file, engine='openpyxl')
tactics_df.sort_values(by='ID', inplace=True)

tactics_df.head()

,ID,STIX ID,name,description,url,created,last modified,domain,version
8,TA0001,x-mitre-tactic--ffd5bcee-6e16-4dd2-8eca-7b3bee...,Initial Access,The adversary is trying to get into your netwo...,https://attack.mitre.org/tactics/TA0001,17 October 2018,25 April 2025,enterprise-attack,1.0
5,TA0002,x-mitre-tactic--4ca45d45-df4d-4613-8980-bac22d...,Execution,The adversary is trying to run malicious code....,https://attack.mitre.org/tactics/TA0002,17 October 2018,25 April 2025,enterprise-attack,1.0
10,TA0003,x-mitre-tactic--5bc1d813-693e-4823-9961-abf9af...,Persistence,The adversary is trying to maintain their foot...,https://attack.mitre.org/tactics/TA0003,17 October 2018,25 April 2025,enterprise-attack,1.0
11,TA0004,x-mitre-tactic--5e29b093-294e-49e9-a803-dab3d7...,Privilege Escalation,The adversary is trying to gain higher-level p...,https://attack.mitre.org/tactics/TA0004,17 October 2018,25 April 2025,enterprise-attack,1.0
14,TA0005,x-mitre-tactic--78b23412-0651-46d7-a540-170a1c...,Stealth,The adversary is trying to hide and conceal th...,https://attack.mitre.org/tactics/TA0005,17 October 2018,12 May 2026,enterprise-attack,1.0


In [28]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(llm_model, trust_remote_code=True, device = device)
tactics_df['embedding'] = tactics_df['description'].apply(lambda x: model.encode(x).tolist())
tactics_df.head()

/root/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning:

Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`



,ID,STIX ID,name,description,url,created,last modified,domain,version,embedding
8,TA0001,x-mitre-tactic--ffd5bcee-6e16-4dd2-8eca-7b3bee...,Initial Access,The adversary is trying to get into your netwo...,https://attack.mitre.org/tactics/TA0001,17 October 2018,25 April 2025,enterprise-attack,1.0,"[0.06018930673599243, 0.0038062790408730507, -..."
5,TA0002,x-mitre-tactic--4ca45d45-df4d-4613-8980-bac22d...,Execution,The adversary is trying to run malicious code....,https://attack.mitre.org/tactics/TA0002,17 October 2018,25 April 2025,enterprise-attack,1.0,"[0.054608069360256195, 0.009688958525657654, -..."
10,TA0003,x-mitre-tactic--5bc1d813-693e-4823-9961-abf9af...,Persistence,The adversary is trying to maintain their foot...,https://attack.mitre.org/tactics/TA0003,17 October 2018,25 April 2025,enterprise-attack,1.0,"[0.05528368055820465, 0.02232680656015873, -0...."
11,TA0004,x-mitre-tactic--5e29b093-294e-49e9-a803-dab3d7...,Privilege Escalation,The adversary is trying to gain higher-level p...,https://attack.mitre.org/tactics/TA0004,17 October 2018,25 April 2025,enterprise-attack,1.0,"[0.058327797800302505, -0.024653282016515732, ..."
14,TA0005,x-mitre-tactic--78b23412-0651-46d7-a540-170a1c...,Stealth,The adversary is trying to hide and conceal th...,https://attack.mitre.org/tactics/TA0005,17 October 2018,12 May 2026,enterprise-attack,1.0,"[-0.01420659851282835, -0.005842362996190786, ..."


In [ ]:

# umap visualization
import umap

umap_model = umap.UMAP(n_neighbors=5, n_components=3, metric='cosine', n_jobs=-1, random_state=42)
embedding_3d = umap_model.fit_transform(list(tactics_df['embedding']))

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [30]:
# plot
import plotly.express as px
import plotly.io as pio
import matplotlib.cm as cm

pio.renderers.default = "vscode" # or "notebook" if you're using Jupyter Notebook

fig = px.scatter_3d(
    embedding_3d,
    x=0,
    y=1,
    z=2,
    hover_data={'name': tactics_df['name'], 'ID': tactics_df['ID']},
    color = list(range(len(tactics_df))),
)
fig.update_layout(title='Tactics Embeddings UMAP Visualization')
fig.show()